[![Fixel Algorithms](https://i.imgur.com/AqKHVZ0.png)](https://fixelalgorithms.gitlab.io)

# AI Program

## Machine Learning - UnSupervised Learning - Clustering - Hierarchical Clustering

This notebook focuses on _Agglomerative Clustering_ (Bottom Up).

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 13/06/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2024_02/0060ClusteringHierarchical.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Machine Learning
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.datasets import load_wine
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

# Miscellaneous
from platform import python_version
import random

# Typing
from typing import Callable, Dict, List, Literal, Optional, Self, Set, Tuple, Union
from numpy.typing import NDArray

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Jupyter
from IPython import get_ipython
from ipywidgets import Dropdown, IntSlider, Layout
from ipywidgets import interact

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

# GMM / K-Means Warnings
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings('ignore', category = ConvergenceWarning) #<! Suppress GMM convergence warning
warnings.filterwarnings('ignore', message = 'KMeans is known to have a memory leak on Windows with MKL'); #<! Suppress KMeans memory leak warning on Windows with MKL
warnings.filterwarnings('ignore', message = 'Found Intel OpenMP'); #<! Suppress different OpenMP library warning
warnings.filterwarnings('ignore', category = RuntimeWarning) #<! Suppress different OpenMP library warning

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

In [ ]:
# Courses Packages

from DataVisualization import PlotDendrogram

In [ ]:
# General Auxiliary Functions


## Clustering Wine Data by Agglomerative (Bottom Up) Policy

In this note book we'll use the Agglomerative method for clustering the _Wine Recognition_ data set.  
We'll use SciKit Learn's [`AgglomerativeClustering`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html) class and visualize the clustering result as a feature.

The _Wine_ data set is useful for this demo as it is small enough to inspect and large enough to show the effect of scaling, linkage method and number of clusters.

* <font color='brown'>(**#**)</font> The labels of the Wine data are known. Yet, they are used only for visualization / validation and not by the clustering algorithm.
* <font color='brown'>(**#**)</font> The "magic" in those method is the definition of the relation between samples and sub sets of samples.

In [ ]:
# Parameters

# Model
numClusters   = 3
linkageMethod = 'ward'

# Dendrogram
valP   = 40
thrLvl = 10

# Visualization
lFeature = ['alcohol', 'flavanoids', 'color_intensity', 'proline']

## Generate / Load Data

The data is based on SciKit Learn's _Wine Recognition_ data set.

The data includes 178 samples and 13 numeric features extracted from chemical analysis of wines from 3 cultivars.

* <font color='brown'>(**#**)</font> See [`load_wine()`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html) for the data set description.
* <font color='brown'>(**#**)</font> In clustering, the labels are not used for training. They are kept only for visualization and external validation.

In [ ]:
# Load Data

dData  = load_wine(as_frame = True)
dfData = dData.frame
dfX    = dData.data
vY     = dData.target
lClass = list(dData.target_names)

print(f'The features data shape: {dfX.shape}')

In [ ]:
# The Data Frame

dfData.head(10)

In [ ]:
# Data Set Information

print(f'Feature Names: {dData.feature_names}')
print(f'Class Names  : {lClass}')

### Plot Data

In [ ]:
# Plot the Data
# Pair Plot of selected features

dfPlot          = dfX[lFeature].copy()
dfPlot['Class'] = pd.Categorical.from_codes(vY, lClass)

sns.pairplot(dfPlot, vars = lFeature, hue = 'Class', height = 3, plot_kws = {'s': 20});

## Pre Process Data

In [ ]:
# Standardize Data

oStdScaler = StandardScaler()
mX         = oStdScaler.fit_transform(dfX)

dfZ = pd.DataFrame(mX, columns = dfX.columns)
dfZ.head(10)

## Cluster Data by SciKit Learn Agglomerative Clustering

The algorithm works as following:

1. Set ${\color{magenta}\mathcal{C}}$, the set of all clusters: 

$$
{\color{magenta}\mathcal{C}}=\left\{{\color{green} \left\{ \boldsymbol{x}_{1}\right\}} ,{\color{green}\left\{ \boldsymbol{x}_{2}\right\}} ,\dots,{\color{green}\left\{ \boldsymbol{x}_{N}\right\}} \right\} 
$$

2. While $\left| {\color{magenta}\mathcal{C}} \right| > K$:
   - Set ${\color{green}\mathcal{C}_{i^{\star}}},{\color{green}\mathcal{C}_{j^{\star}}}\leftarrow\arg\min_{{\color{green}\mathcal{C}_{i}},{\color{green}\mathcal{C}_{j}}\in{\color{magenta}\mathcal{C}}}d_{\mathcal{C}}\left({\color{green}\mathcal{C}_{i}},{\color{green}\mathcal{C}_{j}}\right)$.
   - Set ${\color{green}\widetilde{\mathcal{C}}}\leftarrow{\color{green}\mathcal{C}_{i^{\star}}}\cup{\color{green}\mathcal{C}_{j^{\star}}}$.
   - Set ${\color{magenta}\mathcal{C}}\leftarrow{\color{magenta}\mathcal{C}}\backslash\left\{ {\color{green}\mathcal{C}_{i^{\star}}},{\color{green}\mathcal{C}_{j^{\star}}}\right\} $.
   - Set ${\color{magenta}\mathcal{C}}\leftarrow{\color{magenta}\mathcal{C}}\cup{\color{green}\widetilde{C}}$.

The _Hyper Parameters_ of the model are:

1. The number of clusters $K$.
2. The clusters dissimilarity function / linkage method.

* <font color='brown'>(**#**)</font> The [`AgglomerativeClustering`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html) class exposes the labels using `fit_predict()` or the `labels_` attribute.
* <font color='brown'>(**#**)</font> The `ward` linkage minimizes the increase in within cluster variance.

In [ ]:
# Build and Train the Model

oAggCluster = AgglomerativeClustering(n_clusters = numClusters, linkage = linkageMethod)
vL          = oAggCluster.fit_predict(mX)

* <font color='red'>(**?**)</font> In the context of new data, what is the limitation of `AgglomerativeClustering` compared to models which implement `predict()`?

In [ ]:
# Add the Cluster ID as a Feature

dfXX          = dfX[lFeature].copy()
dfXX['Class'] = pd.Categorical.from_codes(vY, lClass)
dfXX['Label'] = vL

In [ ]:
# Feature Analysis

sns.pairplot(dfXX, vars = lFeature, hue = 'Label', palette = sns.color_palette()[:numClusters], height = 3, plot_kws = {'s': 20});

## Compare to the Known Classes

The Wine labels are not used by the clustering model.  
Yet they can be used to inspect how well the unsupervised clusters align with the known cultivars.

* <font color='brown'>(**#**)</font> Cluster IDs have arbitrary order. A cluster label of `0` does not necessarily correspond to class `0`.

### External Validation

In [ ]:
# Compare Known Classes and Cluster Labels

dfConf = pd.crosstab(dfXX['Class'], dfXX['Label'], rownames = ['Wine Class'], colnames = ['Cluster'])
dfConf

### Rand Index

The **Rand Index (RI)** compares 2 partitions of the same samples by looking at all pairs of samples.  
For every pair it asks whether the 2 partitions agree:

1. The pair is in the same group in both partitions.
2. The pair is in different groups in both partitions.

If $a$ is the number of pairs placed together by both partitions and $b$ is the number of pairs separated by both partitions, then:

$$
\mathrm{RI}=\frac{a+b}{\binom{N}{2}}
$$

The score is between $0$ and $1$, where $1$ means the partitions agree perfectly.  
For clustering, this is useful because it does not require cluster IDs to have the same names as the true classes. It only checks whether pairs of samples are grouped consistently.

The issue with the plain Rand Index is that it includes agreement that happens **by chance**.  
This is especially visible in clustering, because many pairs of samples are usually in different clusters. Even 2 unrelated partitions may agree on many of those separated pairs, so the RI can look high even when the clustering has little meaningful relation to the true classes.

The **Adjusted Rand Index (ARI)** applies a chance correction:

$$
\mathrm{ARI}=\frac{\mathrm{RI}-\mathbb{E}\left[\mathrm{RI}\right]}{\mathrm{RI}_{\max}-\mathbb{E}\left[\mathrm{RI}\right]}
$$

The correction does 2 things:

1. It subtracts the expected Rand Index obtained from random cluster assignments with the same cluster sizes.
2. It rescales the result so perfect agreement remains equal to $1$.

Equivalently, ARI measures how much better the clustering is than random agreement, relative to the best possible improvement above random.  
Hence:

* $\mathrm{ARI}=1$: Perfect agreement.
* $\mathrm{ARI}\approx0$: Agreement is similar to random labeling.
* $\mathrm{ARI}<0$: Worse than random agreement.

This makes ARI a better external validation score than RI when comparing clustering labels to known classes.

* <font color='red'>(**?**)</font> Explain the matching results in the context of **Adjusted Rand Index (ARI)**.

In [ ]:
# Quantitative Score

scoreAri = adjusted_rand_score(vY, vL)

print(f'The Adjusted Rand Index (ARI) score: {scoreAri:0.3f}')

## Dendrogram Visualization

The `AgglomerativeClustering` model gives the cluster labels directly.  
To visualize the hierarchy, we can use SciPy's dendrogram on the same standardized features.

* <font color='brown'>(**#**)</font> The `y` values stand for the cost of merging 2 clusters (The distance between them).  
  Later merges have higher values as the linkage functions produce a monotonic hierarchy.

In [ ]:
# Plot the Dendrogram

hA = PlotDendrogram(dfZ, linkageMethod, valP, thrLvl, figSize = (10, 6))
hA.set_title(f'Wine Data Dendrogram, Linkage Method: {linkageMethod.title()}');

### Interactive Visualization

In [ ]:
# Interactive Visualization

hPlotDendrogram = lambda linkageMethod, thrLvl: PlotDendrogram(dfZ, linkageMethod, valP, thrLvl, figSize = (10, 6))

linkageMethodDropdown = Dropdown(description = 'Linkage Method', options = [('Single', 'single'), ('Complete', 'complete'), ('Average', 'average'), ('Ward', 'ward')], value = linkageMethod, layout = Layout(width = 'max-content'), style = {'description_width': '120px'})
thrLvlSlider          = IntSlider(description = 'Threshold', min = 1, max = 30, step = 1, value = thrLvl, layout = Layout(width = '30%'), style = {'description_width': '120px'})

interact(hPlotDendrogram, linkageMethod = linkageMethodDropdown, thrLvl = thrLvlSlider);

## Selecting the Number of Clusters by a Global Method

A useful global method for choosing the number of clusters is the **Gap Statistic**.  
Unlike AIC / BIC, it is not tied to a probabilistic model.  
Unlike the Elbow method, it includes a formal comparison against a reference distribution.

The idea is to compare the clustering quality on the real data to the clustering quality expected from data with **no cluster structure**.  
For each candidate number of clusters $K$:

1. Cluster the real data and compute its within cluster dispersion ${W}_{K}$.
2. Generate several synthetic data sets from a uniform distribution over the same feature values range.
3. Cluster each synthetic data set using the same clustering method and compute its dispersion.
4. Compare the real data dispersion to the average synthetic dispersion (Reference):

$$
\mathrm{Gap} \left( K \right) = \mathbb{E}_{\mathrm{ref}} \left[ \log \left( {W}_{K}^{\mathrm{ref}} \right) \right] - \log \left( {W}_{K} \right)
$$

A large gap means the real data has much tighter clusters than random unstructured data.  
The usual selection rule chooses the smallest $K$ such that:

$$
\mathrm{Gap} \left( K \right) \geq \mathrm{Gap} \left( K + 1 \right) - s_{K + 1}
$$

Where ${s}_{K + 1} = \sqrt{1 + \frac{1}{B}} \cdot \operatorname{std}_{b = 1, \ldots, B}
\left( \log \left( W_{K, b}^{\mathrm{ref}} \right) \right)$ accounts for the variability across the synthetic data sets.

This method is model agnostic: it only needs the algorithm to produce cluster labels for each $K$.  
Therefore, the same procedure can be applied to `KMeans`, `GaussianMixture` and `AgglomerativeClustering`.

* <font color='brown'>(**#**)</font> A common choice for the definition of dispersion is the sum of squared distances of samples from their assigned cluster centroids.

In [ ]:
# Functions

def CalcDispersion(mD: NDArray, vLabel: NDArray) -> float:
    # Sum of squared distances from samples to their assigned cluster centers
    
    valDisp = 0.0

    for valLbl in np.unique(vLabel):
        mC       = mD[vLabel == valLbl]
        vCenter  = np.mean(mC, axis = 0)
        valDisp += np.sum(np.square(mC - vCenter))

    return valDisp


def CalcGapStatistic(mD: NDArray, vK: NDArray, hCluster: Callable[[NDArray, int], NDArray], *, numRef: int, seedNum: int) -> pd.DataFrame:
    oRng  = np.random.default_rng(seedNum)
    vMin  = np.min(mD, axis = 0)
    vMax  = np.max(mD, axis = 0)
    lRows = []

    for k in vK:
        vLabel  = hCluster(mD, k)
        valLogW = np.log(CalcDispersion(mD, vLabel))
        vLogWRef = np.empty(numRef)

        for ii in range(numRef):
            mRef         = oRng.uniform(vMin, vMax, size = mD.shape) #<! Synthetic data generated uniformly within the range of the original data
            vLabelRef    = hCluster(mRef, k)
            vLogWRef[ii] = np.log(CalcDispersion(mRef, vLabelRef))

        valGap = np.mean(vLogWRef) - valLogW
        valStd = np.std(vLogWRef, ddof = 1) * np.sqrt(1 + (1 / numRef)) #<! The value of ${s}_k$ is the standard deviation of the simulated reference dispersion values, adjusted for the number of reference datasets used in the simulation. The adjustment factor $\sqrt{1 + \frac{1}{B}}$ accounts for the variability introduced by using a finite number of reference datasets (B). This adjustment ensures that the standard deviation reflects the uncertainty in estimating the expected dispersion under the null hypothesis, providing a more accurate measure of variability when comparing the observed dispersion to the reference distribution.

        lRows.append({'K': k, 'Gap': valGap, 'Std': valStd})

    return pd.DataFrame(lRows)


def GetOptimalK(dfGap: pd.DataFrame) -> int:
    for ii in range(len(dfGap) - 1):
        gapK      = dfGap.loc[ii, 'Gap']
        gapNext   = dfGap.loc[ii + 1, 'Gap']
        stdNext   = dfGap.loc[ii + 1, 'Std']

        if gapK >= gapNext - stdNext:
            return int(dfGap.loc[ii, 'K'])

    return int(dfGap.iloc[-1]['K'])

In [ ]:
# Gap Statistic on the Wine Data

vK     = np.arange(1, 9)
numRef = 25

dClusterMethod = {
    'K-Means': lambda mD, k: KMeans(n_clusters = k, n_init = 10, random_state = seedNum).fit_predict(mD),
    'GMM': lambda mD, k: GaussianMixture(n_components = k, covariance_type = 'full', random_state = seedNum).fit_predict(mD),
    'Agglomerative': lambda mD, k: AgglomerativeClustering(n_clusters = k, linkage = linkageMethod).fit_predict(mD),
}

lDfGap       = []
lSelectedRow = []

for methodName, hCluster in dClusterMethod.items():
    dfGapMethod = CalcGapStatistic(mX, vK, hCluster, numRef = numRef, seedNum = seedNum)
    optK        = GetOptimalK(dfGapMethod)

    dfGapMethod['Method'] = methodName
    lDfGap.append(dfGapMethod)
    lSelectedRow.append({'Method': methodName, 'Selected K': optK})

dfGapAll       = pd.concat(lDfGap, ignore_index = True)
dfGapSelection = pd.DataFrame(lSelectedRow)
dfGapSelected  = pd.merge(dfGapAll, dfGapSelection, left_on = ['Method', 'K'], right_on = ['Method', 'Selected K'])

hF, hA = plt.subplots(figsize = (10, 6))
sns.lineplot(data = dfGapAll, x = 'K', y = 'Gap', hue = 'Method', marker = 'o', ax = hA)

for methodName, dfMethod in dfGapAll.groupby('Method', sort = False):
    hA.errorbar(dfMethod['K'], dfMethod['Gap'], yerr = dfMethod['Std'], fmt = 'none', capsize = 4, alpha = 0.6)

sns.scatterplot(data = dfGapSelected, x = 'K', y = 'Gap', hue = 'Method', marker = 'X', s = 180, legend = False, ax = hA)

hA.set_title('Gap Statistic by Number of Clusters')
hA.set_xlabel('Number of Clusters')
hA.set_ylabel('Gap Statistic')
hA.set_xticks(vK)
hA.grid(True);

In [ ]:
display(dfGapAll.pivot(index = 'K', columns = 'Method', values = 'Gap').round(3))
display(dfGapSelection)